# 12 Aligned Transformer + XGBoost Evaluation

This notebook evaluates the separate hybrid model that first learns an aligned multimodal representation using a structured transformer and note encoder, then trains XGBoost on the exported aligned embedding plus tabular features.

## Evaluation scope

- keep all existing notebook 08 outputs untouched
- evaluate only artifacts stored under `10_aligned_transformer_xgboost`
- summarize both the neural alignment encoder and the final hybrid XGBoost model
- write results into a separate processed stage

In [1]:
import sys
from pathlib import Path

NOTEBOOK_CWD = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        candidate
        for candidate in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents]
        if (candidate / 'configs' / 'base.yaml').exists() and (candidate / 'src').is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not locate the project root from the current notebook directory.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.runtime import load_project_runtime

runtime = load_project_runtime(start=PROJECT_ROOT)
IN_COLAB = runtime.in_colab
PROJECT_ROOT = runtime.project_root
config = runtime.config
paths = runtime.paths

PROJECT_ROOT


PosixPath('/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy')

In [2]:
if IN_COLAB:
    %pip -q install pyyaml pandas numpy scikit-learn


In [3]:
import pandas as pd

from src.evaluation.artifacts import build_lead_time_table, summarize_predictions
from src.utils.io_utils import save_dataframe_bundle
from src.utils.logging_utils import write_run_manifest


In [4]:
{
    'aligned_transformer_xgboost': config['aligned_transformer_xgboost'],
    'aligned_transformer_xgboost_evaluation': config['aligned_transformer_xgboost_evaluation'],
    'evaluation': config['evaluation'],
}


{'aligned_transformer_xgboost': {'output_stage': '10_aligned_transformer_xgboost',
  'structured_encoder': 'transformer',
  'text_encoder_mode': 'frozen_embedding',
  'hidden_dim': 128,
  'aligned_dim': 256,
  'dropout': 0.2,
  'batch_size': 8,
  'epochs': 10,
  'learning_rate': 0.0003,
  'weight_decay': 0.0001,
  'gradient_clip_norm': 1.0,
  'scheduler': 'cosine',
  'min_learning_rate': 5e-05,
  'early_stopping_patience': 4,
  'threshold_selection_metric': 'f1',
  'max_structured_steps': 48,
  'max_text_windows': 8,
  'max_tokens_per_window': 128,
  'text_embedding_dim': 768,
  'structured_aggregations': ['mean', 'min', 'max', 'last'],
  'include_missingness': True,
  'include_static_categoricals': True,
  'include_note_metadata': True,
  'include_clinical_event_features': True,
  'clinical_event_lookback_hours': 48,
  'xgboost_model': 'xgboost'},
 'aligned_transformer_xgboost_evaluation': {'output_stage': '12_aligned_transformer_xgboost_evaluation'},
 'evaluation': {'default_threshol

## Collect hybrid prediction tables

In [5]:
model_output_dir = paths['processed_data_dir'] / config.get('aligned_transformer_xgboost', {}).get('output_stage', '10_aligned_transformer_xgboost')
evaluation_output_dir = paths['processed_data_dir'] / config.get('aligned_transformer_xgboost_evaluation', {}).get('output_stage', '12_aligned_transformer_xgboost_evaluation')

prediction_files = sorted(model_output_dir.glob('*_predictions.csv'))
assert prediction_files, 'No aligned Transformer + XGBoost prediction files found. Run scripts/train_aligned_transformer_xgboost_local.py first.'

pd.DataFrame({
    'prediction_file': [path.name for path in prediction_files],
    'source_stage': [path.parent.name for path in prediction_files],
}).head(20)


,prediction_file,source_stage
0,horizon_12h_aligned_transformer_encoder_test_p...,10_aligned_transformer_xgboost
1,horizon_12h_aligned_transformer_encoder_val_pr...,10_aligned_transformer_xgboost
2,horizon_12h_aligned_transformer_xgboost_test_p...,10_aligned_transformer_xgboost
3,horizon_12h_aligned_transformer_xgboost_val_pr...,10_aligned_transformer_xgboost
4,horizon_24h_aligned_transformer_encoder_test_p...,10_aligned_transformer_xgboost
5,horizon_24h_aligned_transformer_encoder_val_pr...,10_aligned_transformer_xgboost
6,horizon_24h_aligned_transformer_xgboost_test_p...,10_aligned_transformer_xgboost
7,horizon_24h_aligned_transformer_xgboost_val_pr...,10_aligned_transformer_xgboost
8,horizon_6h_aligned_transformer_encoder_test_pr...,10_aligned_transformer_xgboost
9,horizon_6h_aligned_transformer_encoder_val_pre...,10_aligned_transformer_xgboost


## Evaluate each prediction file

In [6]:
MODEL_FAMILY_MAP = {
    'aligned_transformer_encoder': 'alignment_encoder',
    'aligned_transformer_xgboost': 'hybrid_xgboost',
}

metric_rows = []
artifact_bundle = {}

for path in prediction_files:
    predictions_df = pd.read_csv(path)
    model_name = predictions_df['model_name'].iloc[0] if 'model_name' in predictions_df.columns else path.stem
    dataset_name = predictions_df['dataset_name'].iloc[0] if 'dataset_name' in predictions_df.columns else 'unknown_dataset'
    split = 'test' if path.name.endswith('_test_predictions.csv') else 'val' if path.name.endswith('_val_predictions.csv') else 'unknown'
    model_family = MODEL_FAMILY_MAP.get(model_name, 'other')
    horizon_hours = int(dataset_name.split('_')[1].replace('h', '')) if dataset_name.startswith('horizon_') and len(dataset_name.split('_')) >= 2 else -1

    metrics, curves = summarize_predictions(
        predictions_df=predictions_df,
        threshold=config['evaluation']['default_threshold'],
        calibration_bins=config['evaluation']['calibration_bins'],
    )
    metric_rows.append({
        'dataset_name': dataset_name,
        'model_name': model_name,
        'model_family': model_family,
        'source_stage': path.parent.name,
        'prediction_file': path.name,
        'split': split,
        **metrics,
    })

    lead_time_df = build_lead_time_table(predictions_df, horizon_hours=horizon_hours)
    artifact_bundle[f'{dataset_name}_{model_name}_{split}_roc_curve'] = curves['roc_curve']
    artifact_bundle[f'{dataset_name}_{model_name}_{split}_pr_curve'] = curves['pr_curve']
    artifact_bundle[f'{dataset_name}_{model_name}_{split}_calibration'] = curves['calibration']
    artifact_bundle[f'{dataset_name}_{model_name}_{split}_lead_time'] = lead_time_df

evaluation_df = pd.DataFrame(metric_rows).sort_values(['dataset_name', 'model_name', 'split']).reset_index(drop=True)
artifact_bundle['aligned_transformer_xgboost_evaluation_summary'] = evaluation_df
evaluation_df


,dataset_name,model_name,model_family,source_stage,prediction_file,split,auroc,auprc,accuracy,precision,recall,f1,brier_score,specificity,sensitivity,tp,fp,tn,fn,decision_threshold
0,horizon_12h,aligned_transformer_encoder,alignment_encoder,10_aligned_transformer_xgboost,horizon_12h_aligned_transformer_encoder_test_p...,test,0.744065,0.163577,0.929618,0.234043,0.276151,0.253359,0.041242,0.959153,0.276151,66,216,5072,173,0.027812
1,horizon_12h,aligned_transformer_encoder,alignment_encoder,10_aligned_transformer_xgboost,horizon_12h_aligned_transformer_encoder_val_pr...,val,0.711375,0.124823,0.930584,0.175373,0.231527,0.199575,0.035767,0.957728,0.231527,47,221,5007,156,0.027812
2,horizon_12h,aligned_transformer_xgboost,hybrid_xgboost,10_aligned_transformer_xgboost,horizon_12h_aligned_transformer_xgboost_test_p...,test,0.998574,0.960815,0.993667,0.901575,0.958159,0.929006,0.005310,0.995272,0.958159,229,25,5263,10,0.175241
3,horizon_12h,aligned_transformer_xgboost,hybrid_xgboost,10_aligned_transformer_xgboost,horizon_12h_aligned_transformer_xgboost_val_pr...,val,0.998807,0.965827,0.994844,0.910798,0.955665,0.932692,0.004690,0.996366,0.955665,194,19,5209,9,0.175241
4,horizon_24h,aligned_transformer_encoder,alignment_encoder,10_aligned_transformer_xgboost,horizon_24h_aligned_transformer_encoder_test_p...,test,0.745187,0.146782,0.940518,0.241935,0.171429,0.200669,0.040978,0.975540,0.171429,30,94,3749,145,0.062811
5,horizon_24h,aligned_transformer_encoder,alignment_encoder,10_aligned_transformer_xgboost,horizon_24h_aligned_transformer_encoder_val_pr...,val,0.718422,0.117677,0.948461,0.245763,0.198630,0.219697,0.034653,0.976889,0.198630,29,89,3762,117,0.062811
6,horizon_24h,aligned_transformer_xgboost,hybrid_xgboost,10_aligned_transformer_xgboost,horizon_24h_aligned_transformer_xgboost_test_p...,test,0.999236,0.978877,0.995520,0.928962,0.971429,0.949721,0.003340,0.996617,0.971429,170,13,3830,5,0.319207
7,horizon_24h,aligned_transformer_xgboost,hybrid_xgboost,10_aligned_transformer_xgboost,horizon_24h_aligned_transformer_xgboost_val_pr...,val,0.998229,0.952739,0.994246,0.891720,0.958904,0.924092,0.005494,0.995586,0.958904,140,17,3834,6,0.319207
8,horizon_6h,aligned_transformer_encoder,alignment_encoder,10_aligned_transformer_xgboost,horizon_6h_aligned_transformer_encoder_test_pr...,test,0.774473,0.218083,0.917453,0.260526,0.331104,0.291605,0.050412,0.949168,0.331104,99,281,5247,200,0.016365
9,horizon_6h,aligned_transformer_encoder,alignment_encoder,10_aligned_transformer_xgboost,horizon_6h_aligned_transformer_encoder_val_pre...,val,0.785994,0.265255,0.917668,0.262755,0.358885,0.303387,0.046950,0.947050,0.358885,103,289,5169,184,0.016365


## Inspect strongest hybrid results

In [7]:
summary_columns = [
    'dataset_name',
    'split',
    'model_family',
    'model_name',
    'auroc',
    'auprc',
    'accuracy',
    'precision',
    'recall',
    'f1',
    'decision_threshold',
]

hybrid_only_df = (
    evaluation_df.loc[evaluation_df['model_name'] == 'aligned_transformer_xgboost', summary_columns]
    .sort_values(['dataset_name', 'split', 'auprc'], ascending=[True, True, False])
    .reset_index(drop=True)
)
encoder_only_df = (
    evaluation_df.loc[evaluation_df['model_name'] == 'aligned_transformer_encoder', summary_columns]
    .sort_values(['dataset_name', 'split', 'auprc'], ascending=[True, True, False])
    .reset_index(drop=True)
)

display(hybrid_only_df)
display(encoder_only_df)


,dataset_name,split,model_family,model_name,auroc,auprc,accuracy,precision,recall,f1,decision_threshold
0,horizon_12h,test,hybrid_xgboost,aligned_transformer_xgboost,0.998574,0.960815,0.993667,0.901575,0.958159,0.929006,0.175241
1,horizon_12h,val,hybrid_xgboost,aligned_transformer_xgboost,0.998807,0.965827,0.994844,0.910798,0.955665,0.932692,0.175241
2,horizon_24h,test,hybrid_xgboost,aligned_transformer_xgboost,0.999236,0.978877,0.995520,0.928962,0.971429,0.949721,0.319207
3,horizon_24h,val,hybrid_xgboost,aligned_transformer_xgboost,0.998229,0.952739,0.994246,0.891720,0.958904,0.924092,0.319207
4,horizon_6h,test,hybrid_xgboost,aligned_transformer_xgboost,0.998627,0.974953,0.995195,0.932907,0.976589,0.954248,0.250000
5,horizon_6h,val,hybrid_xgboost,aligned_transformer_xgboost,0.998984,0.977741,0.994256,0.934932,0.951220,0.943005,0.250000


,dataset_name,split,model_family,model_name,auroc,auprc,accuracy,precision,recall,f1,decision_threshold
0,horizon_12h,test,alignment_encoder,aligned_transformer_encoder,0.744065,0.163577,0.929618,0.234043,0.276151,0.253359,0.027812
1,horizon_12h,val,alignment_encoder,aligned_transformer_encoder,0.711375,0.124823,0.930584,0.175373,0.231527,0.199575,0.027812
2,horizon_24h,test,alignment_encoder,aligned_transformer_encoder,0.745187,0.146782,0.940518,0.241935,0.171429,0.200669,0.062811
3,horizon_24h,val,alignment_encoder,aligned_transformer_encoder,0.718422,0.117677,0.948461,0.245763,0.198630,0.219697,0.062811
4,horizon_6h,test,alignment_encoder,aligned_transformer_encoder,0.774473,0.218083,0.917453,0.260526,0.331104,0.291605,0.016365
5,horizon_6h,val,alignment_encoder,aligned_transformer_encoder,0.785994,0.265255,0.917668,0.262755,0.358885,0.303387,0.016365


## Save evaluation artifacts

In [8]:
saved_paths = save_dataframe_bundle(artifact_bundle, evaluation_output_dir)
manifest_path = paths['manifests_dir'] / '12_aligned_transformer_xgboost_evaluation_manifest.json'
write_run_manifest(
    path=manifest_path,
    stage='12_aligned_transformer_xgboost_evaluation',
    config=config,
    extra={
        'source_stage': str(model_output_dir),
        'saved_artifacts': saved_paths,
    },
)

saved_paths, manifest_path


({'horizon_12h_aligned_transformer_encoder_test_roc_curve': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy/results/processed/12_aligned_transformer_xgboost_evaluation/horizon_12h_aligned_transformer_encoder_test_roc_curve.csv',
  'horizon_12h_aligned_transformer_encoder_test_pr_curve': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy/results/processed/12_aligned_transformer_xgboost_evaluation/horizon_12h_aligned_transformer_encoder_test_pr_curve.csv',
  'horizon_12h_aligned_transformer_encoder_test_calibration': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy/results/processed/12_aligned_transformer_xgboost_evaluation/horizon_12h_aligned_transformer_encoder_test_calibration.csv',
  'horizon_12h_aligned_transformer_encoder_test_lead_time': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy/results/processed/12_aligned_transformer_xgboost_evaluation/horizon_12h_aligned_transformer_encoder_test_lead_time.csv',
  'horizon_12h_aligned_transformer_enc